# UE Data Mining - Projet

Notebook à compléter dans le cadre du projet de l’UE Data Mining des Master 2 Intelligence Artificielle et Master 2 Data Science de l’année scolaire 2026-2027.

* Nom du groupe : 12
* Membres du groupe :
    * idil MAHAMOUD GUEDI
    * André da Conceição
    * Ali Chabaka 
    * Aissam Eddine Boukhelkhal
    * Lucas Duchamp
    * RATOANDROMANANA Iavosoa Miella


In [1]:
import pandas as pd

In [129]:
# Lecture des fichiers csv
df_public_appointments = pd.read_csv('data/public_records/public_appointments.csv')
df_registered_entities = pd.read_csv('data/public_records/registered_entities.csv')
df_regulatory_case_events = pd.read_csv('data/public_records/regulatory_case_events.csv')
df_site_access_case_events = pd.read_csv('data/public_records/site_access_case_events.csv')

## Nettoyage de public_appointments

In [98]:
df_public_appointments.dtypes

appointment_record_id          str
appointing_jurisdiction        str
office_holder_name             str
recorded_birth_year        float64
office_title                   str
institution_name               str
appointment_start              str
appointment_end                str
appointment_basis              str
departure_reason               str
official_region                str
source_notice_reference        str
dtype: object

In [100]:
# Recherche de doublons
print("Doublons:",df_public_appointments.duplicated().sum())

Doublons: 0


Pas de doublons dans le fichier public_appointments

In [101]:
# Recherche de données manquantes
print("Données manquantes dans chaque colonne")
print(df_public_appointments.isna().sum())

Données manquantes dans chaque colonne
appointment_record_id       0
appointing_jurisdiction     0
office_holder_name          0
recorded_birth_year         9
office_title                0
institution_name            0
appointment_start           0
appointment_end            34
appointment_basis           0
departure_reason           34
official_region             0
source_notice_reference     0
dtype: int64


On constate qu'il manque des données dans recorded_birth_year,appointment_end et departure_reason

In [102]:
# Conversion des valeurs dans recorded_birth_year en int et remplacements des valeurs manquantes par None
df_public_appointments["recorded_birth_year"] = df_public_appointments["recorded_birth_year"].fillna(0).apply('int64')

# Conversion des valeurs manquantes par None
for c in ["appointment_end", "departure_reason"]:
    df_public_appointments[c] = df_public_appointments[c].fillna("Unknown")
print("Valeurs manquantes dans chaque colonne:")
print(df_public_appointments.isna().sum())

Valeurs manquantes dans chaque colonne:
appointment_record_id      0
appointing_jurisdiction    0
office_holder_name         0
recorded_birth_year        0
office_title               0
institution_name           0
appointment_start          0
appointment_end            0
appointment_basis          0
departure_reason           0
official_region            0
source_notice_reference    0
dtype: int64


In [103]:
# Conversion des dates dans la colonne appointment_start dans le format Y-M-D
df_public_appointments["appointment_start"] = pd.to_datetime(df_public_appointments["appointment_start"], dayfirst=True, format="mixed")

df_public_appointments.head(7)

,appointment_record_id,appointing_jurisdiction,office_holder_name,recorded_birth_year,office_title,institution_name,appointment_start,appointment_end,appointment_basis,departure_reason,official_region,source_notice_reference
0,APT-000001,Aveloran,Ilyan Rho,1975,Regional pollinator and agricultural technical...,Avelora Agricultural Resilience Agency,2003-01-01,Unknown,appointed,Unknown,Aveloran,NOTICE-AVL-2003-0002
1,APT-000002,Kintaran,Amara Senn,1980,Cave-access programme manager,Kintara Cave and Pollinator Authority,2005-01-01,Unknown,appointed,Unknown,Kintaran,NOTICE-KNT-2005-0003
2,APT-000003,Kintaran,Leora Kassen,1947,Head of government of Kintara Republic,Office of the Kintara Chancellor,1978-01-01,Unknown,appointed,Unknown,Kintaran,NOTICE-KNT-1978-0004
3,APT-000004,Aveloran,Tavian Serat,1953,Senior minister of the Avelora Federation,Avelora Ministry of Strategic Resources,1984-01-01,Unknown,appointed,Unknown,Aveloran,NOTICE-AVL-1984-0005
4,APT-000005,Tembaran,Mirek Talven,1942,Regional governor in the Tembara Commonwealth,Tembara Coastal Administration,1974-01-01,Unknown,appointed,Unknown,Tembaran,NOTICE-TMB-1974-0006
5,APT-000006,Aveloran,Elira Vaun,0,Retired political intermediary and former coun...,Avelora Council of State,1965-01-01,2021-09-18,appointed,deceased,Aveloran,NOTICE-AVL-1965-0007
6,APT-000007,Valennan,Dr. Lysa Quen,1968,Senior medical auditor,Orison Medical Audit Board,1995-01-01,Unknown,appointed,Unknown,Valennan,NOTICE-VLN-1995-0008


In [104]:
# Encodage catégoriel des colonnes:
# appointing_jurisdiction, institution_name, official_region, departure_reason
dummies_juridisc = pd.get_dummies(df_public_appointments["appointing_jurisdiction"], prefix="appoint_juridisd", dtype=int)
dummies_inst_name = pd.get_dummies(df_public_appointments["institution_name"],prefix="inst_name", dtype=int)
dummies_official_reg = pd.get_dummies(df_public_appointments["official_region"], prefix="off_reg", dtype=int)
dummies_depart_reason = pd.get_dummies(df_public_appointments["departure_reason"], prefix="dep_reason",dtype=int)

# Concaténation de l'encodage 
df_public_appointments_clean = pd.concat([df_public_appointments, dummies_juridisc,dummies_inst_name,dummies_official_reg,dummies_depart_reason], axis=1)

df_public_appointments_clean.head(10)

,appointment_record_id,appointing_jurisdiction,office_holder_name,recorded_birth_year,office_title,institution_name,appointment_start,appointment_end,appointment_basis,departure_reason,...,inst_name_Tembara Biodiversity Service,inst_name_Tembara Coastal Administration,off_reg_Aveloran,off_reg_Kintaran,off_reg_Tembaran,off_reg_Valennan,dep_reason_Unknown,dep_reason_deceased,dep_reason_retired,dep_reason_role_transition
0,APT-000001,Aveloran,Ilyan Rho,1975,Regional pollinator and agricultural technical...,Avelora Agricultural Resilience Agency,2003-01-01,Unknown,appointed,Unknown,...,0,0,1,0,0,0,1,0,0,0
1,APT-000002,Kintaran,Amara Senn,1980,Cave-access programme manager,Kintara Cave and Pollinator Authority,2005-01-01,Unknown,appointed,Unknown,...,0,0,0,1,0,0,1,0,0,0
2,APT-000003,Kintaran,Leora Kassen,1947,Head of government of Kintara Republic,Office of the Kintara Chancellor,1978-01-01,Unknown,appointed,Unknown,...,0,0,0,1,0,0,1,0,0,0
3,APT-000004,Aveloran,Tavian Serat,1953,Senior minister of the Avelora Federation,Avelora Ministry of Strategic Resources,1984-01-01,Unknown,appointed,Unknown,...,0,0,1,0,0,0,1,0,0,0
4,APT-000005,Tembaran,Mirek Talven,1942,Regional governor in the Tembara Commonwealth,Tembara Coastal Administration,1974-01-01,Unknown,appointed,Unknown,...,0,1,0,0,1,0,1,0,0,0
5,APT-000006,Aveloran,Elira Vaun,0,Retired political intermediary and former coun...,Avelora Council of State,1965-01-01,2021-09-18,appointed,deceased,...,0,0,1,0,0,0,0,1,0,0
6,APT-000007,Valennan,Dr. Lysa Quen,1968,Senior medical auditor,Orison Medical Audit Board,1995-01-01,Unknown,appointed,Unknown,...,0,0,0,0,0,1,1,0,0,0
7,APT-000008,Kintaran,Leora Kassen,1947,Minister for Rural Development,Kintara Cave and Pollinator Authority,1978-01-04,1987-10-31,appointed,role_transition,...,0,0,0,1,0,0,0,0,0,1
8,APT-000009,Kintaran,Leora Kassen,1947,Cabinet coordinator,Office of the Kintara Chancellor,1988-01-15,1999-12-31,appointed,role_transition,...,0,0,0,1,0,0,0,0,0,1
9,APT-000010,Aveloran,Tavian Serat,1953,Agricultural policy secretary,Avelora Agricultural Resilience Agency,1984-01-02,1994-05-31,appointed,role_transition,...,0,0,1,0,0,0,0,0,0,1


## Nettoyage de registered entities

In [117]:
print("Les types des colonnes sont:",df_registered_entities.dtypes)
df_registered_entities.shape

Les types des colonnes sont: registration_number               str
legal_name                        str
registered_entity_type            str
registration_date                 str
current_status                    str
registered_address                str
jurisdiction                      str
declared_activity                 str
parent_registration_number    float64
last_filing_date                  str
dtype: object


(24, 10)

In [118]:
# Recherche de doublons
print("Doublons:", df_registered_entities.duplicated().sum())

Doublons: 0


On constate qu'on a aucun doublons.

In [119]:
# Recherche de données manquantes
print("Données manquantes dans chaque colonne")
print(df_registered_entities.isna().sum())

Données manquantes dans chaque colonne
registration_number            0
legal_name                     0
registered_entity_type         0
registration_date              0
current_status                 0
registered_address             0
jurisdiction                   0
declared_activity              0
parent_registration_number    24
last_filing_date               0
dtype: int64


On constate que la colonne parent_registration_number est entièrement vide.

In [120]:
# Retirer la colonne vide
df_registered_entities.dropna(inplace=True, axis=1, how='all')
df_registered_entities.head(10)

,registration_number,legal_name,registered_entity_type,registration_date,current_status,registered_address,jurisdiction,declared_activity,last_filing_date
0,REG-VLN-01001,Meridian BioLogistics,logistics_company,1996-01-15,active,Meridian Valenne Headquarters,Valenne Republic,International biological and cold-chain logistics,2025-03-31
1,REG-VLN-01002,Asterion Clinical Institute,research_institute,1989-01-15,active,Asterion Central Clinical Campus,Valenne Republic,Private preventive-medicine and research insti...,2025-03-31
2,REG-TMB-01003,Northstar Cold Chain Services,logistics_company,1992-01-15,active,Northstar Tembara Cold-Chain Hub,Tembara Commonwealth,Refrigerated transport provider,2025-03-31
3,REG-VLN-01004,Whitehaven Strategic Holdings,consulting_company,1987-01-15,active,Meridian Valenne Headquarters,Valenne Republic,Investment and consulting company,2025-03-31
4,REG-VLN-01005,Valenne Industrial Futures Council,trade_association,1984-01-15,active,Valenne Central District,Valenne Republic,Industry-policy association,2025-03-31
5,REG-VLN-01006,Vale Strategy Collective,consulting_company,2008-01-15,active,Vale Strategy Valenne Office,Valenne Republic,Climate and social-policy consultancy,2025-03-31
6,REG-AVL-01007,Threshold Earth Observatory,research_institute,2011-01-15,active,Threshold Earth Avelora Laboratory,Avelora Federation,Environmental research organization,2025-03-31
7,REG-VLN-01008,Common Ground Climate Forum,activist_organization,2013-01-15,active,Valenne Central District,Valenne Republic,Climate and economic-justice coalition,2025-03-31
8,REG-AVL-01009,Pollinator Justice Network,activist_organization,2014-01-15,active,Eastern Avelora Agricultural Cluster,Avelora Federation,Agricultural and ecological advocacy group,2025-03-31
9,REG-VLN-01010,Lantern Investigative Bureau,media_organization,2015-01-15,active,Valenne Central District,Valenne Republic,Independent journalism and research group,2025-03-31


In [121]:
# Encodage catégoriel des colonnes suivantes:
# registered_entity_type, current_status, jurisdiction
dummies_registered_e_type = pd.get_dummies(df_registered_entities["registered_entity_type"],prefix="entity_type",dtype=int)
dummies_current_status = pd.get_dummies(df_registered_entities["current_status"],prefix="current_s",dtype=int)
dummies_jurisdiction = pd.get_dummies(df_registered_entities["jurisdiction"],prefix="juridisc",dtype=int)
df_reg_entities_clean = pd.concat([df_registered_entities,dummies_registered_e_type, dummies_current_status, dummies_jurisdiction], axis=1)

df_reg_entities_clean.head(5)

,registration_number,legal_name,registered_entity_type,registration_date,current_status,registered_address,jurisdiction,declared_activity,last_filing_date,entity_type_activist_organization,...,entity_type_media_organization,entity_type_research_institute,entity_type_trade_association,current_s_active,current_s_dissolved,juridisc_Avelora Federation,juridisc_Kintara Republic,juridisc_Nordhaven Union,juridisc_Tembara Commonwealth,juridisc_Valenne Republic
0,REG-VLN-01001,Meridian BioLogistics,logistics_company,1996-01-15,active,Meridian Valenne Headquarters,Valenne Republic,International biological and cold-chain logistics,2025-03-31,0,...,0,0,0,1,0,0,0,0,0,1
1,REG-VLN-01002,Asterion Clinical Institute,research_institute,1989-01-15,active,Asterion Central Clinical Campus,Valenne Republic,Private preventive-medicine and research insti...,2025-03-31,0,...,0,1,0,1,0,0,0,0,0,1
2,REG-TMB-01003,Northstar Cold Chain Services,logistics_company,1992-01-15,active,Northstar Tembara Cold-Chain Hub,Tembara Commonwealth,Refrigerated transport provider,2025-03-31,0,...,0,0,0,1,0,0,0,0,1,0
3,REG-VLN-01004,Whitehaven Strategic Holdings,consulting_company,1987-01-15,active,Meridian Valenne Headquarters,Valenne Republic,Investment and consulting company,2025-03-31,0,...,0,0,0,1,0,0,0,0,0,1
4,REG-VLN-01005,Valenne Industrial Futures Council,trade_association,1984-01-15,active,Valenne Central District,Valenne Republic,Industry-policy association,2025-03-31,0,...,0,0,1,1,0,0,0,0,0,1


## Nettoyage de regulatory_case_events

In [130]:
df_regulatory_case_events.dtypes

regulatory_event_id            str
case_number                    str
case_category                  str
event_time                     str
activity                       str
subject_name                   str
subject_registration_number    str
agency_name                    str
initiating_source              str
assigned_unit                  str
related_site_code              str
related_shipment_number        str
previous_status                str
new_status                     str
outcome                        str
public_reference               str
dtype: object

In [131]:
df_regulatory_case_events.shape

(233, 16)

In [132]:
#Recherche de doublons
print("Nombre de doublons:",df_regulatory_case_events.duplicated().sum())

Nombre de doublons: 1


On constate qu'il y a un doublon. On va donc supprimer la ligne duppliquée.

In [133]:
df_regulatory_case_events_d = df_regulatory_case_events.drop_duplicates().reset_index(drop=True)
print("Nombre de doublons:",df_regulatory_case_events_d.duplicated().sum())

Nombre de doublons: 0


In [134]:
# Recherche de données manquantes
print("Données manquantes dans chaque colonne")
print(df_regulatory_case_events_d.isna().sum())

Données manquantes dans chaque colonne
regulatory_event_id              0
case_number                      0
case_category                    0
event_time                       0
activity                         0
subject_name                     0
subject_registration_number     73
agency_name                      0
initiating_source                0
assigned_unit                    0
related_site_code              110
related_shipment_number        140
previous_status                  0
new_status                       0
outcome                          0
public_reference               155
dtype: int64


On constate qu'il manque des données dans les colonnes subject_registration_number, related_site_code, related_shipment_number, public_reference

In [135]:
# Clés relationnelles
df_regulatory_case_events_d["has_related_site_code"] = df_regulatory_case_events_d["related_site_code"].notna().astype(int)
df_regulatory_case_events_d["has_related_shipment_number"] = df_regulatory_case_events_d["related_shipment_number"].notna().astype(int)

# Conversions des valeurs manquantes
df_regulatory_case_events_d["related_site_code"] = df_regulatory_case_events_d["related_site_code"].fillna("None")
df_regulatory_case_events_d["related_shipment_number"] = df_regulatory_case_events_d["related_shipment_number"].fillna("None")

df_regulatory_case_events_d["subject_registration_number"] = df_regulatory_case_events_d["subject_registration_number"].fillna("Unknown")
df_regulatory_case_events_d["public_reference"] = df_regulatory_case_events_d["public_reference"].fillna("Unknown")

df_regulatory_case_events_d.head(10)

,regulatory_event_id,case_number,case_category,event_time,activity,subject_name,subject_registration_number,agency_name,initiating_source,assigned_unit,related_site_code,related_shipment_number,previous_status,new_status,outcome,public_reference,has_related_site_code,has_related_shipment_number
0,RGE-000001,REG-2018-0001,environmental_complaint,2018-05-14T09:00:00,case_opened,Avelora Agricultural Resilience Agency,Unknown,Tembara Customs Coordination Unit,civil society referral,integrity review,AVL-MAR,SHP-2018-00032,new,open,case registered,Unknown,1,1
1,RGE-000002,REG-2018-0001,environmental_complaint,2018-05-18T09:00:00,evidence_received,Avelora Agricultural Resilience Agency,Unknown,Tembara Customs Coordination Unit,civil society referral,integrity review,AVL-MAR,SHP-2018-00032,open,evidence_review,material received from Lantern Investigative B...,Unknown,1,1
2,RGE-000003,REG-2018-0001,environmental_complaint,2018-05-20T09:00:00,triage_review,Avelora Agricultural Resilience Agency,Unknown,Tembara Customs Coordination Unit,civil society referral,integrity review,AVL-MAR,SHP-2018-00032,evidence_review,assigned,case assigned,Unknown,1,1
3,RGE-000004,REG-2018-0001,environmental_complaint,2018-05-23T09:00:00,technical_review,Avelora Agricultural Resilience Agency,Unknown,Tembara Customs Coordination Unit,civil society referral,integrity review,AVL-MAR,SHP-2018-00032,assigned,findings_review,technical review completed,Unknown,1,1
4,RGE-000005,REG-2018-0001,environmental_complaint,2018-05-27T09:00:00,finding_issued,Avelora Agricultural Resilience Agency,Unknown,Tembara Customs Coordination Unit,civil society referral,integrity review,AVL-MAR,SHP-2018-00032,findings_review,monitoring,monitoring recommendation issued,Unknown,1,1
5,RGE-000006,REG-2018-0001,environmental_complaint,2018-06-01T09:00:00,case_closed,Avelora Agricultural Resilience Agency,Unknown,Tembara Customs Coordination Unit,civil society referral,integrity review,AVL-MAR,SHP-2018-00032,monitoring,closed,case closed,Unknown,1,1
6,RGE-000007,REG-2019-0002,environmental_complaint,2019-03-08T09:00:00,case_opened,Kintara Cave and Pollinator Authority,Unknown,Tembara Customs Coordination Unit,civil society referral,integrity review,KNT-KES,SHP-2018-00007,new,open,case registered,Unknown,1,1
7,RGE-000008,REG-2019-0002,environmental_complaint,2019-03-12T09:00:00,evidence_received,Kintara Cave and Pollinator Authority,Unknown,Tembara Customs Coordination Unit,civil society referral,integrity review,KNT-KES,SHP-2018-00007,open,evidence_review,material received from Pollinator Justice Network,Unknown,1,1
8,RGE-000009,REG-2019-0002,environmental_complaint,2019-03-14T09:00:00,triage_review,Kintara Cave and Pollinator Authority,Unknown,Tembara Customs Coordination Unit,civil society referral,integrity review,KNT-KES,SHP-2018-00007,evidence_review,assigned,case assigned,Unknown,1,1
9,RGE-000010,REG-2019-0002,environmental_complaint,2019-03-17T09:00:00,technical_review,Kintara Cave and Pollinator Authority,Unknown,Tembara Customs Coordination Unit,civil society referral,integrity review,KNT-KES,SHP-2018-00007,assigned,findings_review,technical review completed,Unknown,1,1


In [136]:
print("Données manquantes dans chaque colonne")
print(df_regulatory_case_events_d.isna().sum())

Données manquantes dans chaque colonne
regulatory_event_id            0
case_number                    0
case_category                  0
event_time                     0
activity                       0
subject_name                   0
subject_registration_number    0
agency_name                    0
initiating_source              0
assigned_unit                  0
related_site_code              0
related_shipment_number        0
previous_status                0
new_status                     0
outcome                        0
public_reference               0
has_related_site_code          0
has_related_shipment_number    0
dtype: int64


In [138]:
# Conversions des dates
df_regulatory_case_events_d["event_time"] = pd.to_datetime(df_regulatory_case_events_d["event_time"],errors="coerce",format="mixed")

df_regulatory_case_events_d.head()

,regulatory_event_id,case_number,case_category,event_time,activity,subject_name,subject_registration_number,agency_name,initiating_source,assigned_unit,related_site_code,related_shipment_number,previous_status,new_status,outcome,public_reference,has_related_site_code,has_related_shipment_number
0,RGE-000001,REG-2018-0001,environmental_complaint,2018-05-14 09:00:00,case_opened,Avelora Agricultural Resilience Agency,Unknown,Tembara Customs Coordination Unit,civil society referral,integrity review,AVL-MAR,SHP-2018-00032,new,open,case registered,Unknown,1,1
1,RGE-000002,REG-2018-0001,environmental_complaint,2018-05-18 09:00:00,evidence_received,Avelora Agricultural Resilience Agency,Unknown,Tembara Customs Coordination Unit,civil society referral,integrity review,AVL-MAR,SHP-2018-00032,open,evidence_review,material received from Lantern Investigative B...,Unknown,1,1
2,RGE-000003,REG-2018-0001,environmental_complaint,2018-05-20 09:00:00,triage_review,Avelora Agricultural Resilience Agency,Unknown,Tembara Customs Coordination Unit,civil society referral,integrity review,AVL-MAR,SHP-2018-00032,evidence_review,assigned,case assigned,Unknown,1,1
3,RGE-000004,REG-2018-0001,environmental_complaint,2018-05-23 09:00:00,technical_review,Avelora Agricultural Resilience Agency,Unknown,Tembara Customs Coordination Unit,civil society referral,integrity review,AVL-MAR,SHP-2018-00032,assigned,findings_review,technical review completed,Unknown,1,1
4,RGE-000005,REG-2018-0001,environmental_complaint,2018-05-27 09:00:00,finding_issued,Avelora Agricultural Resilience Agency,Unknown,Tembara Customs Coordination Unit,civil society referral,integrity review,AVL-MAR,SHP-2018-00032,findings_review,monitoring,monitoring recommendation issued,Unknown,1,1


In [140]:
# Encodage catégoriel des valeurs
# case_category, activity, agency_name,civil society referral, integrity review
dummies_cc = pd.get_dummies(df_regulatory_case_events_d["case_category"],prefix="case_cat", dtype=int)
dummies_a = pd.get_dummies(df_regulatory_case_events_d["activity"], prefix="act", dtype=int)
dummies_agency = pd.get_dummies(df_regulatory_case_events_d["agency_name"], prefix="agency", dtype=int)
dummies_init_source = pd.get_dummies(df_regulatory_case_events_d["initiating_source"], prefix="init_source", dtype=int)
dummies_assigned_unit = pd.get_dummies(df_regulatory_case_events_d["assigned_unit"], prefix="assigned_unit", dtype=int)

# Concaténation finale avec les dummies déjà créées
df_regulatory_case_events_clean = pd.concat(
    [
        df_regulatory_case_events_d,
        dummies_cc,
        dummies_a,
        dummies_agency,
        dummies_init_source,
        dummies_assigned_unit
    ],
    axis=1
)

df_regulatory_case_events_clean.head()


,regulatory_event_id,case_number,case_category,event_time,activity,subject_name,subject_registration_number,agency_name,initiating_source,assigned_unit,...,act_triage_review,agency_Avelora Food Standards Office,agency_Nordhaven Maritime Safety Agency,agency_Orison Medical Audit Board,agency_Tembara Customs Coordination Unit,agency_Valenne Public Procurement Authority,init_source_civil society referral,init_source_routine referral,assigned_unit_case review,assigned_unit_integrity review
0,RGE-000001,REG-2018-0001,environmental_complaint,2018-05-14 09:00:00,case_opened,Avelora Agricultural Resilience Agency,Unknown,Tembara Customs Coordination Unit,civil society referral,integrity review,...,0,0,0,0,1,0,1,0,0,1
1,RGE-000002,REG-2018-0001,environmental_complaint,2018-05-18 09:00:00,evidence_received,Avelora Agricultural Resilience Agency,Unknown,Tembara Customs Coordination Unit,civil society referral,integrity review,...,0,0,0,0,1,0,1,0,0,1
2,RGE-000003,REG-2018-0001,environmental_complaint,2018-05-20 09:00:00,triage_review,Avelora Agricultural Resilience Agency,Unknown,Tembara Customs Coordination Unit,civil society referral,integrity review,...,1,0,0,0,1,0,1,0,0,1
3,RGE-000004,REG-2018-0001,environmental_complaint,2018-05-23 09:00:00,technical_review,Avelora Agricultural Resilience Agency,Unknown,Tembara Customs Coordination Unit,civil society referral,integrity review,...,0,0,0,0,1,0,1,0,0,1
4,RGE-000005,REG-2018-0001,environmental_complaint,2018-05-27 09:00:00,finding_issued,Avelora Agricultural Resilience Agency,Unknown,Tembara Customs Coordination Unit,civil society referral,integrity review,...,0,0,0,0,1,0,1,0,0,1


## Nettoyage de site_access_events

In [141]:
df_site_access_case_events.dtypes

access_event_id                   str
access_case_number                str
site_code                         str
event_time                        str
activity                          str
requesting_organization_name      str
requesting_registration_number    str
authorizing_agency                str
authorized_person_name            str
access_purpose                    str
equipment_category                str
previous_status                   str
new_status                        str
confidentiality_class             str
outcome                           str
dtype: object

In [142]:
# Recherche de doublons dans les événements d'accès au site
print("Doublons dans df_site_access_case_events :", df_site_access_case_events.duplicated().sum())

dup_rows = df_site_access_case_events[df_site_access_case_events.duplicated(keep=False)]
dup_rows.head()

Doublons dans df_site_access_case_events : 1


,access_event_id,access_case_number,site_code,event_time,activity,requesting_organization_name,requesting_registration_number,authorizing_agency,authorized_person_name,access_purpose,equipment_category,previous_status,new_status,confidentiality_class,outcome
167,ACE-000168,ACC-2022-0022,KNT-NNM,2022-11-02T09:00:00,closure_recorded,Kintara Field Equipment Cooperative,REG-KNT-20006,Kintara Cave and Pollinator Authority,Uma Wessen,sensor maintenance,maintenance tools,completed,closed,standard,case closed
412,ACE-000168,ACC-2022-0022,KNT-NNM,2022-11-02T09:00:00,closure_recorded,Kintara Field Equipment Cooperative,REG-KNT-20006,Kintara Cave and Pollinator Authority,Uma Wessen,sensor maintenance,maintenance tools,completed,closed,standard,case closed


In [143]:
# Suppression des doublons
df_site_access_case_events_d = (
    df_site_access_case_events.drop_duplicates().reset_index(drop=True)
)
print("Nombre de doublons :", df_site_access_case_events_d.duplicated().sum())

Nombre de doublons : 0


In [144]:
# Vérification des données manquantes
print("Données manquantes dans chaque colonne :")
print(df_site_access_case_events_d.isna().sum())

Données manquantes dans chaque colonne :
access_event_id                    0
access_case_number                 0
site_code                          0
event_time                         0
activity                           0
requesting_organization_name       0
requesting_registration_number    65
authorizing_agency                 0
authorized_person_name             2
access_purpose                     0
equipment_category                 0
previous_status                    0
new_status                         0
confidentiality_class              0
outcome                            0
dtype: int64


In [149]:
# Indicateur et remplacement des identifiants d'entité manquants
df_site_access_case_events_d["has_requesting_registration_number"] = df_site_access_case_events_d["requesting_registration_number"].notna().astype(int)
df_site_access_case_events_d["requesting_registration_number"] = df_site_access_case_events_d["requesting_registration_number"].fillna("Unknown")

df_site_access_case_events_d["authorized_person_name"] = df_site_access_case_events["authorized_person_name"].fillna("Unknown")


In [150]:
print("Données manquantes dans chaque colonne :")
print(df_site_access_case_events_d.isna().sum())

Données manquantes dans chaque colonne :
access_event_id                       0
access_case_number                    0
site_code                             0
event_time                            0
activity                              0
requesting_organization_name          0
requesting_registration_number        0
authorizing_agency                    0
authorized_person_name                0
access_purpose                        0
equipment_category                    0
previous_status                       0
new_status                            0
confidentiality_class                 0
outcome                               0
has_requesting_registration_number    0
dtype: int64


In [152]:
# Conversion de la date
df_site_access_case_events_d["event_time"] = pd.to_datetime(
    df_site_access_case_events_d["event_time"],
    errors="coerce",
    format="mixed"
)
df_site_access_case_events_d.head()

,access_event_id,access_case_number,site_code,event_time,activity,requesting_organization_name,requesting_registration_number,authorizing_agency,authorized_person_name,access_purpose,equipment_category,previous_status,new_status,confidentiality_class,outcome,has_requesting_registration_number
0,ACE-000001,ACC-2018-0001,TMB-NAC,2018-01-15 09:00:00,request_submitted,Horizon Field Research Cooperative,REG-AVL-01013,Tembara Biodiversity Service,Bela Ferran,acoustic monitoring and equipment maintenance,portable monitoring and cold-storage equipment,draft,submitted,restricted,request logged,1
1,ACE-000002,ACC-2018-0001,TMB-NAC,2018-01-21 09:00:00,technical_review,Horizon Field Research Cooperative,REG-AVL-01013,Tembara Biodiversity Service,Kellan Orell,acoustic monitoring and equipment maintenance,portable monitoring and cold-storage equipment,submitted,technical_review,restricted,technical review completed,1
2,ACE-000003,ACC-2018-0001,TMB-NAC,2018-01-26 09:00:00,wildlife_review,Horizon Field Research Cooperative,REG-AVL-01013,Tembara Biodiversity Service,Jalen Quorin,acoustic monitoring and equipment maintenance,portable monitoring and cold-storage equipment,technical_review,wildlife_review,restricted,wildlife conditions recorded,1
3,ACE-000004,ACC-2018-0001,TMB-NAC,2018-01-30 09:00:00,security_review,Horizon Field Research Cooperative,REG-AVL-01013,Tembara Biodiversity Service,Jalen Zorin,acoustic monitoring and equipment maintenance,portable monitoring and cold-storage equipment,wildlife_review,security_review,restricted,site-security conditions recorded,1
4,ACE-000005,ACC-2018-0001,TMB-NAC,2018-02-05 09:00:00,authorization_granted,Horizon Field Research Cooperative,REG-AVL-01013,Tembara Biodiversity Service,Xara Lask,acoustic monitoring and equipment maintenance,portable monitoring and cold-storage equipment,security_review,authorized,restricted,authorization issued,1


In [158]:
# Majuscule en minuscule
df_site_access_case_events_d["activity"] = df_site_access_case_events_d["activity"].str.lower()

In [159]:


# Encodage des colonnes catégorielles
categorical_columns = [
    "authorizing_agency",
    "access_purpose",
    "equipment_category",
    "confidentiality_class",
]

df_site_access_case_events_d[categorical_columns] = (
    df_site_access_case_events_d[categorical_columns].fillna("Unknown")
)

dummies_site_access = pd.get_dummies(
    df_site_access_case_events_d[categorical_columns],
    prefix=categorical_columns,
    dtype=int
)

# Création du DataFrame nettoyé
df_site_access_case_events_clean = pd.concat(
    [df_site_access_case_events_d, dummies_site_access],
    axis=1
)

df_site_access_case_events_clean.head()

,access_event_id,access_case_number,site_code,event_time,activity,requesting_organization_name,requesting_registration_number,authorizing_agency,authorized_person_name,access_purpose,...,access_purpose_sensor maintenance,equipment_category_acoustic and camera equipment,equipment_category_acoustic monitoring equipment,equipment_category_camera equipment,equipment_category_maintenance tools,equipment_category_portable monitoring and cold-storage equipment,equipment_category_protective field equipment,confidentiality_class_confidential,confidentiality_class_restricted,confidentiality_class_standard
0,ACE-000001,ACC-2018-0001,TMB-NAC,2018-01-15 09:00:00,request_submitted,Horizon Field Research Cooperative,REG-AVL-01013,Tembara Biodiversity Service,Bela Ferran,acoustic monitoring and equipment maintenance,...,0,0,0,0,0,1,0,0,1,0
1,ACE-000002,ACC-2018-0001,TMB-NAC,2018-01-21 09:00:00,technical_review,Horizon Field Research Cooperative,REG-AVL-01013,Tembara Biodiversity Service,Kellan Orell,acoustic monitoring and equipment maintenance,...,0,0,0,0,0,1,0,0,1,0
2,ACE-000003,ACC-2018-0001,TMB-NAC,2018-01-26 09:00:00,wildlife_review,Horizon Field Research Cooperative,REG-AVL-01013,Tembara Biodiversity Service,Jalen Quorin,acoustic monitoring and equipment maintenance,...,0,0,0,0,0,1,0,0,1,0
3,ACE-000004,ACC-2018-0001,TMB-NAC,2018-01-30 09:00:00,security_review,Horizon Field Research Cooperative,REG-AVL-01013,Tembara Biodiversity Service,Jalen Zorin,acoustic monitoring and equipment maintenance,...,0,0,0,0,0,1,0,0,1,0
4,ACE-000005,ACC-2018-0001,TMB-NAC,2018-02-05 09:00:00,authorization_granted,Horizon Field Research Cooperative,REG-AVL-01013,Tembara Biodiversity Service,Xara Lask,acoustic monitoring and equipment maintenance,...,0,0,0,0,0,1,0,0,1,0
